> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업의 실습이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🧹 핵심 속성 추출과 전처리

**목표**: 학습에 쓸 수 있는 깨끗한 표를 만든다.

1. 핵심 속성 고르기 2. 데이터 삭제(연도 열·겹친 행·이상치) 3. 결측치 삭제와 대체(평균값·최빈값) 4. 텍스트 값 대체 5. 크기 맞추기(정규화)

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추(또는 **Shift + Enter**)를 누르면 그 셀이 실행돼요.
> 위에서부터 **차례로** 실행하세요. 고쳐 보고 싶으면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 것으로 만드세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명과 사이트의 의사코드를 짝지어 읽으면 돼요.

- 수업용 펭귄 데이터(345줄)를 불러와요.

In [ ]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins.csv')
print(df.shape)

## 1. 핵심 속성 고르기 — 종마다 값이 다른가?

목표는 **종 맞히기**예요. 종마다 평균이 크게 다른 속성일수록 종을 가르는 데 도움이 돼요.

- groupby('종')은 종마다 묶어서 계산해요. 젠투는 날개가 길고 무겁고, 아델리는 부리가 짧아요.

In [ ]:
df.groupby('종')[['부리길이', '부리깊이', '날개길이', '몸무게']].mean().round(1)

- crosstab은 두 열의 값 조합마다 개수를 세요. 어느 종이든 수컷·암컷이 비슷하게 있어요 → 성별은 핵심 속성이 아니에요.

In [ ]:
# 글자 열은 개수표로 — 성별은 종을 가르지 못해요
pd.crosstab(df['종'], df['성별'])

- 사이트에서 고른 것과 같은 네 속성이에요. 번호·섬·성별·연도는 입력 X에서 빼요.
- 입력에서 빼는 것과 표에서 지우는 것은 달라요. 표에서는 다음 칸에서 연도만 지워요.

In [ ]:
features = ['부리길이', '부리깊이', '날개길이', '몸무게']   # 핵심 속성
target = '종'                                               # 정답
print('입력 속성:', features, '/ 정답:', target)

## 2. 데이터 삭제 — 필요 없는 열, 겹친 행, 잘못된 행

사이트 3-2쪽과 같은 순서예요. **연도**는 어디에도 안 써서 지워요. **번호**는 4단원에서 짝을 찾는 열쇠로 쓰고, **섬·성별**은 다른 목표(예: 성별 맞히기 프로젝트)에 쓸 수 있어 표에 남겨 둬요.

- drop(columns=[...])는 열을 지워요.
- 앞에 df = 를 붙여야 지운 표가 df에 다시 담겨요. 9열이 8열이 돼요.

In [ ]:
df = df.drop(columns=['연도'])            # 열 지우기
print(df.shape)

- duplicated()는 앞에 똑같은 행이 있으면 True예요. drop_duplicates()가 그런 행을 지워요.

In [ ]:
print('겹친 행:', df.duplicated().sum())
df = df.drop_duplicates()                 # 처음 것만 남기고 지우기
print('지운 뒤:', df.shape)

- 가공 시간에 찾은 이상치(8200g)를 인덱스로 지워요.

In [ ]:
s = df['몸무게']
q1, q3 = s.quantile(0.25), s.quantile(0.75)
high = q3 + 1.5 * (q3 - q1)
bad = df[s > high].index                  # 위 울타리보다 큰 행
print('이상치 행:', list(bad))
df = df.drop(index=bad)
print('지운 뒤:', df.shape)

## 3. 결측치 처리 — 지울까, 채울까?

사이트 3-5쪽 끝에서 본 규칙이에요. **측정값이 모두 빈 행은 지우고**(채워도 전부 어림값이라서), **한두 칸만 빈 행은 채워요**(나머지는 진짜 값이라서).

- dropna()는 빈칸이 하나라도 있는 행을 통째로 지워요. 몇 행을 잃는지 먼저 확인해요.
- only_sex는 "성별은 비었고 측정값 네 개는 모두 있는 행"이에요. 입력에 쓰지도 않는 성별 때문에 그 행들까지 잃게 돼요.

In [ ]:
print('빈칸이 있는 행:', df.isnull().any(axis=1).sum())
print('dropna() 하면:', len(df.dropna()), '행만 남아요')
only_sex = df['성별'].isnull() & df[features].notnull().all(axis=1)
print('그중 성별 한 칸만 빈 행:', only_sex.sum())

- subset=features, how='all'은 "네 측정값이 모두 비었을 때만" 지우라는 뜻이에요.
- 2마리(4번, 272번)가 지워져 341행이 돼요.

In [ ]:
# ① 측정값 4개가 모두 빈 행은 채워도 전부 어림값이라 지운다
df = df.dropna(subset=features, how='all')
print(df.shape)
print(df.isnull().sum())

- mean()은 빈칸을 빼고 평균을 구해요.
- fillna(값)은 빈칸을 그 값으로 채워요. 부리길이 빈칸(7번 펭귄) 하나가 채워져요.

In [ ]:
# ② 숫자 열의 빈칸 → 평균값으로 채우기
for col in features:
    m = df[col].mean()
    df[col] = df[col].fillna(m)
    print(f'{col}: 평균 {m:.2f}로 채움')

- value_counts()는 값마다 개수를 세요(사이트의 세기표).
- mode()[0]은 가장 많이 나온 값이에요.

In [ ]:
# ③ 글자 열(성별)의 빈칸 → 최빈값으로 채우기
print(df['성별'].value_counts())
m = df['성별'].mode()[0]
df['성별'] = df['성별'].fillna(m)
print('최빈값:', m)

## 4. 텍스트 값 대체 — 글자를 숫자로

모델은 숫자로 계산해요. **바꿈표(사전)**를 만들어 `map`으로 바꿔요.

- 원래 열은 남겨 두고, 숫자로 바꾼 열을 새로 만들어 견줘 봐요.
- 0·1·2는 이름표일 뿐 크기에 뜻이 없어요.

In [ ]:
sex_map = {'수컷': 0, '암컷': 1}
sp_map = {'아델리': 0, '턱끈': 1, '젠투': 2}
df['성별_숫자'] = df['성별'].map(sex_map)
df['종_숫자'] = df['종'].map(sp_map)
df[['종', '종_숫자', '성별', '성별_숫자']].head()

## ✅ 확인 — 빈칸이 하나도 없나요?

- 0이 나오면 성공! 깨끗한 표(341줄 × 10열)를 penguins_clean.csv로 저장해요.
- 다음 수업들은 미리 인터넷에 올려 둔 같은 341마리 표를 불러와요. 그 표는 숫자로 바꾼 두 열이 없고 연도 열은 남아 있어서 (341, 9)로 보여요. 입력에는 어차피 네 측정값만 써요.

In [ ]:
print(df.isnull().sum().sum(), '개의 빈칸')
print(df.shape)
df.to_csv('penguins_clean.csv', index=False, encoding='utf-8-sig')

## 5. 크기 맞추기(정규화) — 0~1로

사이트 3-7쪽과 같은 일이에요. 몸무게(수천 g)는 다른 속성(수십 mm)보다 숫자가 훨씬 커서, 거리로 재는 모델(k-최근접 이웃·k-평균)에서 거리를 혼자 정해요.
열마다 **(값 − 최솟값) ÷ (최댓값 − 최솟값)**으로 0~1에 맞춰요. 위에서 저장한 df(341, 10)는 그대로 두고, 새 변수 `scaled`에 담아요.

- copy()로 복사본을 만들어 바꾸니 df는 (341, 10) 그대로예요.
- 열마다 가장 작은 값은 0, 가장 큰 값은 1이 돼요. 몸무게 2700~6300g도, 부리깊이 13.1~21.5mm도 모두 0~1이에요.

In [ ]:
scaled = df[features].copy()              # df는 그대로 두고 복사본만 바꿔요
for col in features:
    lo, hi = scaled[col].min(), scaled[col].max()
    scaled[col] = (scaled[col] - lo) / (hi - lo)
print('정규화 전'); print(df[features].agg(['min', 'max']))
print('정규화 뒤'); print(scaled.agg(['min', 'max']))